# RAG Minecraft

## Configuration Gemini API

In [ ]:
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

from langchain_core.prompts import PromptTemplate
from langchain_core.documents import Document
from langchain_community.document_loaders import WebBaseLoader
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import format_document
from langchain_core.runnables import RunnablePassthrough
from langchain_community.vectorstores import Chroma
import shutil
import os
import csv
import requests
from urllib.parse import unquote
from bs4 import BeautifulSoup
from langchain_text_splitters import RecursiveCharacterTextSplitter
from IPython.display import Markdown
import cloudscraper
import time


In [ ]:
from api_config import configure_google_api_key

GOOGLE_API_KEY = configure_google_api_key()

## Scrapping

Congig sources

In [ ]:
WIKI_PAGES = [
    "Minecraft"
]

FANDOM_URLS = [
    "https://minecraft.fandom.com/fr/wiki/Survie",
    "https://minecraft.fandom.com/fr/wiki/Cr%C3%A9atif",
    "https://minecraft.fandom.com/fr/wiki/Hardcore",
    "https://minecraft.fandom.com/fr/wiki/Fabrication",
    "https://minecraft.fandom.com/fr/wiki/Cuisson",
    "https://minecraft.fandom.com/fr/wiki/Alchimie",
    "https://minecraft.fandom.com/fr/wiki/La_Foire_aux_Questions",
    "https://minecraft.fandom.com/fr/wiki/Tutoriels/Choses_%C3%A0_ne_PAS_faire",
    "https://minecraft.fandom.com/fr/wiki/Tutoriels/Survie_dans_le_Nether",
    "https://minecraft.fandom.com/fr/wiki/Tutoriels/L%27End_et_l%27Ender_Dragon",
    "https://minecraft.fandom.com/fr/wiki/Cr%C3%A9atures"
]

In [ ]:
scraper = cloudscraper.create_scraper()

In [ ]:
def write_csv(file_name, paragraphs):
    with open(file_name, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["text"])
        for p in paragraphs:
            writer.writerow([p])

In [ ]:
def write_csv(file_name, paragraphs):
    # Create parent folder automatically if it does not exist
    folder = os.path.dirname(file_name)
    if folder:
        os.makedirs(folder, exist_ok=True)

    with open(file_name, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["text"])
        for p in paragraphs:
            writer.writerow([p])

Wikipedia

In [ ]:
def scrape_wikipedia(title: str):

    url = "https://fr.wikipedia.org/w/api.php"

    params = {
        "action": "parse",
        "page": title,
        "format": "json",
        "prop": "text",
        "redirects": True
    }

    r = requests.get(url, params=params, headers={"User-Agent": "Mozilla/5.0"})
    data = r.json()

    html = data["parse"]["text"]["*"]
    soup = BeautifulSoup(html, "html.parser")

    texts = []
    capture = False

    for tag in soup.find_all(["h2", "p"]):

        t = tag.get_text(" ", strip=True)

        if tag.name == "h2" and "Références" in t:
            break

        if tag.name == "h2":
            capture = True
            continue

        if capture and t:
            texts.append(t)

    write_csv("files/wikipedia.csv", texts)

    return Document(
        page_content="\n\n".join(texts),
        metadata={"source": f"wikipedia:{title}"}
    )

Fandom

In [ ]:
def scrape_fandom(url: str):

    headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0 Safari/537.36"
    ),
    "Accept-Language": "fr-FR,fr;q=0.9",
    "Referer": "https://www.google.com/"
    }

    r = scraper.get(url, headers=headers)

    soup = BeautifulSoup(r.text, "html.parser")

    content = soup.find("div", {"class": "mw-parser-output"})
    if not content:
        return None

    texts = []

    for tag in content.find_all(["p", "h2", "h3", "li"]):

        t = tag.get_text(" ", strip=True)

        if not t:
            continue

        if len(t) < 20:
            continue

        if "Voir aussi" in t or "Notes et références" in t:
            break

        texts.append(t)

    paragraphs = [p.strip() for p in texts if p.strip()]
    page_name = url.split("/wiki/")[-1]
    page_name = unquote(page_name)
    file_name = f"files/{page_name}.csv"

    write_csv(file_name, paragraphs)


    return Document(
        page_content="\n\n".join(texts),
        metadata={"source": url}
    )

Build dataset

In [ ]:
# ==========================================
# RECONSTRUCTION DE L'INDEX DEPUIS LES CSV LOCAUX (Anti-Cloudflare / Offline)
# ==========================================
import os
import csv
from pathlib import Path
from langchain_core.documents import Document

docs = []
files_dir = Path("./files")

print("🔄 Chargement des documents en mode offline...\n")
for csv_path in files_dir.rglob("*.csv"):
    relative_path = csv_path.relative_to(files_dir)
    if csv_path.name == "wikipedia.csv":
        source = "wikipedia:Minecraft"
    else:
        parts = list(relative_path.parts)
        parts[-1] = csv_path.stem
        source = f"https://minecraft.fandom.com/fr/wiki/{'/'.join(parts)}"
        
    paragraphs = []
    try:
        with open(csv_path, "r", encoding="utf-8") as f:
            reader = csv.reader(f)
            next(reader, None)  # Skip header
            for row in reader:
                if row and row[0].strip():
                    paragraphs.append(row[0].strip())
        if paragraphs:
            docs.append(Document(page_content="\n\n".join(paragraphs), metadata={"source": source}))
            print(f"✅ Chargé : {relative_path} ({len(paragraphs)} paragraphes)")
    except Exception as e:
        print(f"❌ Erreur lors de la lecture de {csv_path} : {e}")

print(f"\n📚 TOTAL DOCS CHARGÉS : {len(docs)}")

Chunking

In [ ]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=2000,
    chunk_overlap=150
)

split_docs = splitter.split_documents(docs)

print("CHUNKS:", len(split_docs))

Enrchich

In [ ]:
def enrich_with_source(docs):
    enriched = []

    for d in docs:
        src = d.metadata.get("source", "unknown")

        source_type = "WIKIPEDIA" if "wikipedia" in src else "FANDOM"

        d.page_content = (
            f"SOURCE: {src}\n"
            f"SOURCE_TYPE: {source_type}\n\n"
            f"{d.page_content}"
        )

        enriched.append(d)

    return enriched

split_docs = enrich_with_source(split_docs)

## Initialize embedding model + Retriever

In [ ]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

gemini_embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-2",
    task_type="retrieval_document"
)

#### Store data using chroma

In [ ]:
import os
import shutil
path = "./chroma_minecraft_db"

# On ne supprime que si on veut forcer la reconstruction depuis le Notebook (par défaut on évite pour ne pas casser la base rebuild_db)
force_rebuild_in_notebook = False

if force_rebuild_in_notebook and os.path.exists(path):
    try:
        shutil.rmtree(path)
        print("✅ Ancienne base supprimée dans le notebook.")
    except PermissionError:
        print("dossier encore utilisé → restart kernel puis relance")
else:
    print("ℹ️ Saut de la suppression de la base (base préservée ou inexistante).")


In [ ]:
persist_dir = "./chroma_minecraft_db"
import os

# On ne reconstruit la base dans le Notebook que si elle n'existe pas déjà
if not os.path.exists(persist_dir) or (locals().get('force_rebuild_in_notebook', False)):
    print("🚀 Base vectorielle inexistante ou force_rebuild activé. Ingestion en cours (cela peut prendre 1-2 minutes)...")
    vectorstore = Chroma(
        persist_directory=persist_dir,
        embedding_function=gemini_embeddings
    )
    batch_size = 10
    for i in range(0, len(split_docs), batch_size):
        batch = split_docs[i:i+batch_size]
        vectorstore.add_documents(batch)
        time.sleep(10)
    print("🎉 Ingestion terminée avec succès !")
else:
    print("✅ Base vectorielle détectée ! Saut de l'ingestion lente pour éviter les conflits d'écriture et de verrou SQLite.")

# 2. Reload propre (pour usage RAG)
vectorstore_disk = Chroma(
    persist_directory=persist_dir,
    embedding_function=gemini_embeddings
)

# 3. Retriever FINAL utilisé par le RAG
retriever = vectorstore_disk.as_retriever(
    search_kwargs={"k": 6}
)

# Pour la compatibilité avec les cellules suivantes qui utilisent 'vectorstore' pour compter
vectorstore = vectorstore_disk


In [ ]:
# Check the number of chunks generated after text splitting.
print("split_docs:", len(split_docs))

# Check the number of documents actually stored in the Chroma vector database.
# If this number is equal to len(split_docs), the database was built without missing or duplicated chunks.
print("chroma:", vectorstore._collection.count())

#### Create a retriever using Chroma

In [ ]:
print(retriever.invoke("Minecraft")[0].page_content[:500])

def retrieve_documents(retriever, query):
    if hasattr(retriever, "invoke"):
        return retriever.invoke(query)
    if hasattr(retriever, "get_relevant_documents"):
        return retriever.get_relevant_documents(query)
    raise AttributeError("Le retriever ne supporte ni invoke ni get_relevant_documents.")

## Generator

#### Initialize Generator

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

#### Create prompt templates

In [ ]:
# Prompt template （more strict） to query Gemini
llm_prompt_template = """Tu es un assistant expert sur le jeu Minecraft.
Réponds à la question en utilisant UNIQUEMENT le contexte fourni ci-dessous.
Si la réponse ne se trouve pas dans le contexte ou si tu n'es pas sûr, n'invente rien. Dis EXACTEMENT : "Je suis désolé, mais l'information n'est pas dans les documents fournis."
Sois concis (5 phrases maximum) et cite la ou les sources [SOURCE_TYPE: source] à la fin de ta réponse si tu as trouvé l'information.

Question: {question}
Contexte: {context}
Réponse:"""

llm_prompt = PromptTemplate.from_template(llm_prompt_template)

print(llm_prompt)

#### Create a stuff documents chain

In [ ]:
# Combine data from documents to readable string format.
def format_docs(docs):
    formatted_docs = []

    for doc in docs:
        source = doc.metadata.get("source", "unknown")
        content = doc.page_content

        formatted_text = f"[{source}]\n{content}"

        formatted_docs.append(formatted_text)

    return "\n\n".join(formatted_docs)


rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | llm_prompt
    | llm
    | StrOutputParser()
)

### RAG AVANCÉ : ACTIVE RETRIEVAL (RRR)

In [ ]:
def check_if_answered(reponse):
    prompt = f"La réponse suivante indique-t-elle qu'elle ne trouve pas l'information ou qu'elle ne sait pas ? Réponds OUI ou NON.\nRéponse : {reponse}"
    result = llm.invoke(prompt)
    text_content = str(result.content)
    return "OUI" in text_content.strip().upper()

def ask_with_active_retrieval(question):
    print(f"▶ Question posée : {question}")
    
    reponse = rag_chain.invoke(question)
    phrase_echec = "Je suis désolé, mais l'information n'est pas dans les documents fournis." # Phrase exite dans llm_prompt_template，à détecter pour déclencher l'auto-correction
    
    # Auto-correction
    if check_if_answered(reponse):
        print("Information introuvable. Activation de l'Active Retrieval...")
        
        # 1. IA de réécriture (Query Optimizer)
        llm_rewrite = ChatGoogleGenerativeAI(model="gemini-3.5-flash")
        rewrite_prompt = f"""Tu es un expert Minecraft. Un utilisateur a posé cette question : '{question}'. 
        Cette question n'a donné aucun résultat exact dans notre base de données RAG. 
        Réécris cette question sous forme de 2 ou 3 mots-clés très simples et percutants pour optimiser une recherche par similarité vectorielle. 
        Ne renvoie QUE les mots-clés (par exemple : 'Ender Dragon stratégie'), rien d'autre."""
        
        # Générer la nouvelle requête
        from langchain_core.prompts import PromptTemplate
        from langchain_core.output_parsers import StrOutputParser
        rewrite_chain = PromptTemplate.from_template(rewrite_prompt) | llm_rewrite | StrOutputParser()
        nouvelle_requete = rewrite_chain.invoke({})
        print(f"Nouvelle requête optimisée par l'IA : {nouvelle_requete}")
        
        # 2. Relancer la recherche avec les nouveaux mots-clés
        reponse_niveau_2 = rag_chain.invoke(nouvelle_requete)
        
        # 3. Vérifier si la deuxième tentative a réussi
        if phrase_echec not in reponse_niveau_2:
            print("Succès du Niveau 2.")
            return f"*(Auto-correction RRR : Recherche élargie avec les mots-clés '{nouvelle_requete}')*\n\n{reponse_niveau_2}"
        else:
            print("Échec du Niveau 2. L'information n'existe définitivement pas dans la base.")
            return reponse # Renvoie le message de refus standard
            
    # Si le niveau 1 a marché directement
    print("Succès du Niveau 1.")
    return reponse

### Prompt the model

In [ ]:
Markdown(ask_with_active_retrieval("Quel est l'ingrédient de base indispensable pour l'alchimie ?"))

In [ ]:
Markdown(ask_with_active_retrieval("C'est quoi le Warden dans Minecraft ?"))

In [ ]:
Markdown(ask_with_active_retrieval("Comment survivre dans le Nether dans Minecraft ?"))

In [ ]:
Markdown(ask_with_active_retrieval("C'est quoi l'alchimie dans Minecraft?"))

In [ ]:
Markdown(ask_with_active_retrieval("Quels sont les biomes dans Minecraft?"))

In [ ]:
Markdown(rag_chain.invoke("Est ce que je peux miner du diamant avec une pioche en pierre dans Minecraft?"))

In [ ]:
Markdown(ask_with_active_retrieval("Parle moi des boss dans Minecraft?"))

In [ ]:
Markdown(ask_with_active_retrieval("Il y a combien de dimensions dans Minecraft?"))

In [ ]:
### ÉVALUATION DU RAG (LLM-as-a-Judge)

### ÉVALUATION DU RAG (LLM-as-a-Judge)

In [ ]:
# Ground Truth（vérité terrain） Dataset
eval_dataset = [
    {
        "question": "Quels sont les principaux modes de jeu dans Minecraft ?",
        "ground_truth": "Les principaux modes de jeu sont le mode Survie, le mode Créatif et le mode Hardcore."
    },
    {
        "question": "Comment survivre dans le Nether ?",
        "ground_truth": "Il faut éviter d'y dormir car les lits explosent, porter au moins une pièce d'armure en or pour apaiser les Piglins, et utiliser un briquet pour réactiver le portail si nécessaire."
    },
    {
        "question": "Quel est l'ingrédient de base indispensable pour l'alchimie ?",
        "ground_truth": "La verrue du Nether est l'ingrédient de base pour la plupart des potions."
    },
    {
        "question": "Quel est le nom du boss final du jeu ?",
        "ground_truth": "Le boss final est l'Ender Dragon."
    },
    {
        # Trick question: Test if the system honestly refuses to answer
        "question": "Quelles sont les règles du mode Battle Royale dans Minecraft ?",
        "ground_truth": "Je suis désolé, mais l'information n'est pas dans les documents fournis."
    }
]

# Judge Prompt
judge_template = """Tu es un professeur impartial évaluant un système RAG.
Compare la réponse générée par le système avec la réponse idéale (Ground Truth).
Attribue une note de 1 à 5 selon ces critères :
5 = La réponse est parfaite, précise, et correspond exactement au Ground Truth (ou refuse de répondre correctement si c'est un piège).
3 = La réponse est partiellement correcte mais manque de détails.
1 = La réponse est totalement fausse ou contient des hallucinations.

Ne renvoie QUE le chiffre (1, 2, 3, 4 ou 5), aucun autre texte.

Question posée : {question}
Réponse idéale attendue : {ground_truth}
Réponse générée à évaluer : {generated_answer}
Note (1-5) :"""

judge_prompt = PromptTemplate.from_template(judge_template)

# Judge Chain
judge_chain = judge_prompt | llm | StrOutputParser()

total_score = 0
results = []

for i, data in enumerate(eval_dataset):
    print(f"\\nÉvaluation de la question {i+1}/{len(eval_dataset)}...")
    
    # get RAG system answer with active retrieval
    gen_ans = ask_with_active_retrieval(data["question"])
    
    # score with judge model
    score_str = judge_chain.invoke({
        "question": data["question"],
        "ground_truth": data["ground_truth"],
        "generated_answer": gen_ans
    })
    
    score = int(score_str.strip())
        
    total_score += score
    results.append((data["question"], score))
    print(f"Q: {data['question']}")
    print(f"RAG a répondu: {gen_ans}")
    print(f"Score attribué: {score}/5")

# calcul de la note moyenne
average_score = total_score / len(eval_dataset)
print(f"RÉSULTAT FINAL : Note moyenne du RAG = {average_score:.1f} / 5.0")